# Train YOLOv8 Pose on StanfordExtra (Dogs) and Export to ONNX\nThis notebook prepares YOLO pose labels from StanfordExtra annotations, trains a YOLOv8 pose model, and exports an ONNX model for website inference.

In [ ]:
!pip -q install -U ultralytics==8.3.0 onnx onnxruntime

In [ ]:
!wget -nc http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar\n!mkdir -p IMAGES\n!tar -xf images.tar -C IMAGES\n!wget -nc https://github.com/benjiebob/StanfordExtra/raw/master/keypoint_definitions.csv\n# Upload or place stanfordextra_v12.zip in /content first if needed.\n!unzip -o /content/stanfordextra_v12.zip

In [ ]:
import os\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nimport numpy as np\nfrom shutil import copyfile\nfrom ultralytics import YOLO

In [ ]:
ANN_PATH = "StanfordExtra_V12"\nJSON_PATH = os.path.join(ANN_PATH, "StanfordExtra_v12.json")\nwith open(JSON_PATH, "r") as file:\n    json_data = json.load(file)\n\ntrain_ids = np.load(os.path.join(ANN_PATH,"train_stanford_StanfordExtra_v12.npy"))\nval_ids = np.load(os.path.join(ANN_PATH, "test_stanford_StanfordExtra_v12.npy"))

In [ ]:
DATA_DIR = "animal-pose-data"\nTRAIN_IMG_PATH = os.path.join(DATA_DIR, "train", "images")\nTRAIN_LABEL_PATH = os.path.join(DATA_DIR, "train", "labels")\nVALID_IMG_PATH = os.path.join(DATA_DIR, "valid", "images")\nVALID_LABEL_PATH = os.path.join(DATA_DIR, "valid", "labels")\nfor p in [TRAIN_IMG_PATH, TRAIN_LABEL_PATH, VALID_IMG_PATH, VALID_LABEL_PATH]:\n    os.makedirs(p, exist_ok=True)\n\ntrain_json_data = [json_data[idx] for idx in train_ids]\nval_json_data = [json_data[idx] for idx in val_ids]

In [ ]:
IMAGES_DIR = "/content/IMAGES/Images"\nfor data in train_json_data:\n    img_file = data["img_path"]\n    filename = os.path.basename(img_file)\n    copyfile(os.path.join(IMAGES_DIR, img_file), os.path.join(TRAIN_IMG_PATH, filename))\n\nfor data in val_json_data:\n    img_file = data["img_path"]\n    filename = os.path.basename(img_file)\n    copyfile(os.path.join(IMAGES_DIR, img_file), os.path.join(VALID_IMG_PATH, filename))

In [ ]:
CLASS_ID = 0\n\ndef create_yolo_boxes_kpts(img_size, boxes, lm_kpts):\n    IMG_W, IMG_H = img_size\n\n    # Convert visibility 1->2 as requested for visible points in YOLO pose format.\n    vis_ones = np.where(lm_kpts[:, -1] == 1.0)\n    lm_kpts[vis_ones, -1] = 2.0\n\n    # Ensure non-finite values are handled and clamp coordinates to image bounds.\n    lm_kpts = np.nan_to_num(lm_kpts, nan=0.0, posinf=0.0, neginf=0.0)\n    lm_kpts[:, 0] = np.clip(lm_kpts[:, 0], 0, IMG_W)\n    lm_kpts[:, 1] = np.clip(lm_kpts[:, 1], 0, IMG_H)\n\n    # Normalize bbox and keypoints to [0,1].\n    res_box_array = np.array([IMG_W, IMG_H, IMG_W, IMG_H], dtype=np.float32)\n    res_lm_array = np.array([IMG_W, IMG_H], dtype=np.float32)\n\n    norm_kps_per_img = lm_kpts.copy()\n    norm_kps_per_img[:, :2] = norm_kps_per_img[:, :2] / res_lm_array\n    norm_kps_per_img[:, :2] = np.clip(norm_kps_per_img[:, :2], 0.0, 1.0)\n\n    norm_bbox_per_img = boxes.astype(np.float32) / res_box_array\n    norm_bbox_per_img = np.clip(norm_bbox_per_img, 0.0, 1.0)\n\n    # x_c, y_c, w, h from x_min, y_min, w, h\n    yolo_boxes = norm_bbox_per_img.copy()\n    yolo_boxes[:2] = norm_bbox_per_img[:2] + norm_bbox_per_img[2:] / 2.0\n    yolo_boxes = np.clip(yolo_boxes, 0.0, 1.0)\n\n    return yolo_boxes, norm_kps_per_img\n\ndef create_yolo_txt_files(rows, label_path):\n    for data in rows:\n        image_id = os.path.splitext(os.path.basename(data["img_path"]))[0]\n        img_w, img_h = data["img_width"], data["img_height"]\n        landmark_kpts = np.nan_to_num(np.array(data["joints"], dtype=np.float32))\n        bbox = np.array(data["img_bbox"], dtype=np.float32)\n\n        bboxes_yolo, kpts_yolo = create_yolo_boxes_kpts((img_w, img_h), bbox, landmark_kpts)\n\n        txt_file = f"{image_id}.txt"\n        with open(os.path.join(label_path, txt_file), "w") as f:\n            x_c, y_c, box_w, box_h = [round(float(x), 6) for x in bboxes_yolo]\n            kps_flat = [round(float(ele), 6) for ele in kpts_yolo.flatten().tolist()]\n            line = f"{CLASS_ID} {x_c} {y_c} {box_w} {box_h} " + " ".join(map(str, kps_flat))\n            f.write(line)

In [ ]:
create_yolo_txt_files(train_json_data, TRAIN_LABEL_PATH)\ncreate_yolo_txt_files(val_json_data, VALID_LABEL_PATH)\nprint(f"Train images: {len(os.listdir(TRAIN_IMG_PATH))}, labels: {len(os.listdir(TRAIN_LABEL_PATH))}")\nprint(f"Valid images: {len(os.listdir(VALID_IMG_PATH))}, labels: {len(os.listdir(VALID_LABEL_PATH))}")

In [ ]:
@dataclass(frozen=True)\nclass DatasetConfig:\n    IMAGE_SIZE: int = 640\n    BATCH_SIZE: int = 16\n    CLOSE_MOSAIC: int = 10\n    MOSAIC: float = 0.4\n    FLIP_LR: float = 0.0\n\n@dataclass(frozen=True)\nclass TrainingConfig:\n    DATASET_YAML: str = "animal-keypoints.yaml"\n    MODEL: str = "yolov8m-pose.pt"\n    EPOCHS: int = 100\n    KPT_SHAPE: tuple = (24, 3)\n    PROJECT: str = "Animal_Keypoints"\n    NAME: str = "yolov8m_pose_100_epochs"\n    CLASSES_DICT: dict = field(default_factory=lambda: {0: "dog"})\n\ndata_config = DatasetConfig()\ntrain_config = TrainingConfig()

In [ ]:
dataset_yaml = f"""\npath: {Path(DATA_DIR).resolve()}\ntrain: train/images\nval: valid/images\nkpt_shape: [{train_config.KPT_SHAPE[0]}, {train_config.KPT_SHAPE[1]}]\nflip_idx: []\nnames:\n  0: dog\n"""\nwith open(train_config.DATASET_YAML, "w") as f:\n    f.write(dataset_yaml)\nprint(dataset_yaml)

In [ ]:
model = YOLO(train_config.MODEL)\nresults = model.train(\n    data=train_config.DATASET_YAML,\n    imgsz=data_config.IMAGE_SIZE,\n    epochs=train_config.EPOCHS,\n    batch=data_config.BATCH_SIZE,\n    project=train_config.PROJECT,\n    name=train_config.NAME,\n    mosaic=data_config.MOSAIC,\n    close_mosaic=data_config.CLOSE_MOSAIC,\n    fliplr=data_config.FLIP_LR\n)

In [ ]:
best_pt = Path(train_config.PROJECT) / train_config.NAME / "weights" / "best.pt"\nprint(f"Best checkpoint: {best_pt}")\nbest_model = YOLO(str(best_pt))\nmetrics = best_model.val(data=train_config.DATASET_YAML)\nprint(metrics)

In [ ]:
# Export ONNX for website deployment.\n# dynamic=False keeps a fixed input shape, often preferred by browser runtimes.\nonnx_path = best_model.export(\n    format="onnx",\n    imgsz=data_config.IMAGE_SIZE,\n    dynamic=False,\n    simplify=True,\n    opset=12\n)\nprint(f"Exported ONNX: {onnx_path}")

In [ ]:
# Optional: copy ONNX to a predictable filename for upload to your website.\nimport shutil\nfinal_onnx = Path("dog_pose_yolov8.onnx")\nshutil.copy2(onnx_path, final_onnx)\nprint(f"Website model file: {final_onnx.resolve()}")